### BCSE432E Reinforcement Learning Lab: L29+L30
### Lab Assignment 3

### Policy Iteration to optimize Taxi fleet management

### Name: Yashvanth Karunakaran (23BAI1589)

Notebook Link: [https://github.com/Yashvanthk05/RL/blob/main/23BAI1589_LAB-3.ipynb](https://github.com/Yashvanthk05/RL/blob/main/23BAI1589_LAB-3.ipynb)

### AIM:
To implement Policy Iteration to find the best strategy for a taxi to move between different city zones so that it maximizes passenger pickups and minimizes costs over time.

### Problem Description

#### States:
7 Zones (A, B, C, D, E, F, G)
#### Actions: 
From any zone, the taxi can Move to a neighboring zone or Stay in the same zone
#### Rewards:
- Pickup passenger = +20
- move without passenger = -1 (fuel/time)
- no passenger found = -5

+20 (good zone), -5 (bad zone)

### States, Environment, and Parameters

In [1]:
states = ['A', 'B', 'C', 'D', 'E', 'F', 'G']

neighbors = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'G'],
    'F': ['C'],
    'G': ['E']
}

demand = {
    'A': 0.9, 'B': 0.7, 'C': 0.6,
    'D': 0.2, 'E': 0.5, 'F': 0.4, 'G': 0.3
}

gamma = 0.9 

### Actions Function

In [2]:
def get_actions(state):
    return neighbors[state] + ['stay']

### Reward Function

In [3]:
def get_reward(state, action):
    p = demand[state]
    
    if action == 'stay':
        return p * 20 + (1 - p) * (-5)
    else:
        return p * 20 + (1 - p) * (-1)

### Transition Function

In [4]:
def get_transitions(state, action):
    if action == 'stay':
        return [(1.0, state)]
    else:
        return [
            (0.7, action),
            (0.3, state)
        ]

### Initialize Policy and Value Function

In [5]:
policy = {}
V = {}

for s in states:
    policy[s] = get_actions(s)[0]
    V[s] = 0.0

### Policy Evaluation
V(s) = sum over s' of P(s' | s, pi(s)) * [ R(s, pi(s)) + gamma * V(s') ]

In [6]:
def policy_evaluation(policy, V, theta=1e-4):
    while True:
        delta = 0
        
        for s in states:
            v = V[s]
            a = policy[s]
            
            new_v = 0
            for prob, next_state in get_transitions(s, a):
                reward = get_reward(s, a)
                new_v += prob * (reward + gamma * V[next_state])
            
            V[s] = new_v
            delta = max(delta, abs(v - new_v))
        
        if delta < theta:
            break

### Policy Improvement
pi(s) = argmax over a of sum over s' of P(s' | s, a) * [ R(s, a) + gamma * V(s') ]

In [7]:
def policy_improvement(V, policy):
    stable = True
    
    for s in states:
        old_action = policy[s]
        best_action = None
        best_value = -float('inf')
        
        for a in get_actions(s):
            value = 0
            
            for prob, next_state in get_transitions(s, a):
                reward = get_reward(s, a)
                value += prob * (reward + gamma * V[next_state])
            
            if value > best_value:
                best_value = value
                best_action = a
        
        policy[s] = best_action
        
        if old_action != best_action:
            stable = False
    
    return stable

### Policy Iteration Algorithm

In [ ]:
def policy_iteration():
    iteration = 0
    
    while True:
        iteration += 1
        print(f"Iteration {iteration}")
        
        policy_evaluation(policy, V)
        stable = policy_improvement(V, policy)
        
        if stable:
            print("Policy Converged")
            break

### Run Policy Iteration

In [9]:
policy_iteration()

Iteration 1
Iteration 2
Policy Converged!


### Optimal Policy and Value Function

In [10]:
print("Optimal Policy:")
for s in states:
    print(f"Zone {s}: {policy[s]}")

print("\nValue Function:")
for s in states:
    print(f"V({s}) = {round(V[s], 2)}")

Optimal Policy:
Zone A: stay
Zone B: A
Zone C: A
Zone D: B
Zone E: B
Zone F: C
Zone G: E

Value Function:
V(A) = 175.0
V(B) = 169.79
V(C) = 166.92
V(D) = 150.92
V(E) = 159.55
V(F) = 154.19
V(G) = 144.95


### Value Iteration Algorithm

V(s) = max over a of sum over s' of P(s' | s, a) * [ R(s, a) + gamma * V(s') ]

#### Optimal Policy
pi(s) = argmax over a of sum over s' of P(s' | s, a) * [ R(s, a) + gamma * V(s') ]

In [11]:
def value_iteration(theta=1e-4):
    V_vi = {s: 0 for s in states}
    
    while True:
        delta = 0
        
        for s in states:
            v = V_vi[s]
            max_value = -float('inf')
            
            for a in get_actions(s):
                value = 0
                
                for prob, next_state in get_transitions(s, a):
                    reward = get_reward(s, a)
                    value += prob * (reward + gamma * V_vi[next_state])
                
                max_value = max(max_value, value)
            
            V_vi[s] = max_value
            delta = max(delta, abs(v - max_value))
        
        if delta < theta:
            break
    
    policy_vi = {}
    
    for s in states:
        best_action = None
        best_value = -float('inf')
        
        for a in get_actions(s):
            value = 0
            
            for prob, next_state in get_transitions(s, a):
                reward = get_reward(s, a)
                value += prob * (reward + gamma * V_vi[next_state])
            
            if value > best_value:
                best_value = value
                best_action = a
        
        policy_vi[s] = best_action
    
    return V_vi, policy_vi

### Run Value Iteration

In [12]:
V_vi, policy_vi = value_iteration()

print("Value Iteration Policy:")
for s in states:
    print(f"Zone {s}: {policy_vi[s]}")    

Value Iteration Policy:
Zone A: stay
Zone B: A
Zone C: A
Zone D: B
Zone E: B
Zone F: C
Zone G: E


### Intuition

Policy Iteration vs Value Iteration:

Policy Iteration:
- Evaluates and improves policy step-by-step
- Converges faster (fewer iterations)
- Each step is computationally heavier

Value Iteration:
- Directly updates value function
- Simpler to implement
- Takes more iterations but each is faster

Conclusion:
- Both produce the same optimal policy
- Policy Iteration is better for small problems
- Value Iteration is better for large-scale problems

Taxi Insight:
- Taxi prefers high-demand zones (A, B)
- Avoids low-demand zones (D, G)
- Learns to move intelligently instead of random movement